# RapidOCR-vi — huấn luyện model đọc chữ tiếng Việt

Model Latin có sẵn của PP-OCRv3 chỉ có **17/67** nguyên âm có dấu của
tiếng Việt — thiếu hẳn `ă ơ ư` và toàn bộ dấu hỏi ngã nặng. Đem đọc phụ
đề phim ra `"cach an toan nhat cia han"`.

Notebook này huấn luyện lại phần nhận dạng với bảng **236 ký tự** đầy đủ,
rồi xuất ONNX cắm thẳng vào RapidOCR.

**Chạy trên Colab T4**: Runtime → Change runtime type → T4 GPU.

Không train từ đầu mà **tinh chỉnh từ checkpoint Latin có sẵn** — nó đã
biết hình dáng chữ Latin, ta chỉ dạy thêm phần dấu.


## 1. Kiểm GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. Cài PaddlePaddle GPU và PaddleOCR

Paddle chỉ cần lúc HUẤN LUYỆN. Lúc chạy thật chỉ cần `onnxruntime`.


In [ ]:
!pip -q install paddlepaddle-gpu==2.6.1 \
  -i https://www.paddlepaddle.org.cn/packages/stable/cu120/
!pip -q install paddle2onnx onnxruntime rapidocr_onnxruntime
!git clone -q --depth 1 -b release/2.7 \
  https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip -q install -r requirements.txt


## 3. Lấy dữ liệu

Sinh sẵn ở máy bằng `sinh_du_lieu.py` rồi nén lên Drive:

```bash
python sinh_du_lieu.py --video "phim.mp4" --srt "vi.srt" \
       --so 200000 --ra data_vi
tar -czf data_vi.tar.gz data_vi chars_vi.txt
```

Cấu trúc sau khi giải nén trên Colab:

```
/content/data/data_vi/anh/0000000.jpg ...
/content/data/data_vi/nhan_train.txt
/content/data/data_vi/nhan_val.txt
/content/data/chars_vi.txt
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/data
!tar -xzf '/content/drive/MyDrive/data_vi.tar.gz' -C /content/data
!wc -l /content/data/data_vi/nhan_train.txt
!wc -l /content/data/chars_vi.txt
!head -3 /content/data/data_vi/nhan_train.txt


## 4. Checkpoint Latin làm điểm khởi đầu


In [ ]:
!mkdir -p pretrain
%cd pretrain
!wget -q https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/latin_PP-OCRv3_rec_train.tar
!tar -xf latin_PP-OCRv3_rec_train.tar
%cd ..
!ls pretrain/latin_PP-OCRv3_rec_train


In [ ]:
CFG_TEXT = r'''Global:
  use_gpu: true
  epoch_num: 60
  log_smooth_window: 20
  print_batch_step: 50
  save_model_dir: ./output/rec_vi
  save_epoch_step: 5
  eval_batch_step: [0, 1000]
  cal_metric_during_train: true
  pretrained_model: ./pretrain/latin_PP-OCRv3_rec_train/best_accuracy
  character_dict_path: /content/data/chars_vi.txt
  max_text_length: 60
  use_space_char: true

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0005
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 3.0e-05

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform: null
  Backbone:
    name: MobileNetV1Enhance
    scale: 0.5
    last_conv_stride: [1, 2]
    last_pool_type: avg
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 64
            depth: 2
            hidden_dims: 120
            use_guide: true
          Head:
            fc_decay: 0.00001
      - SARHead:
          enc_dim: 512
          max_text_length: 60

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - SARLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc

Train:
  dataset:
    name: SimpleDataSet
    data_dir: /content/data/data_vi/
    ext_op_transform_idx: 1
    label_file_list: [/content/data/data_vi/nhan_train.txt]
    transforms:
      - DecodeImage: {img_mode: BGR, channel_first: false}
      - RecConAug: {prob: 0.5, ext_data_num: 2, image_shape: [48, 320, 3]}
      - RecAug:
      - MultiLabelEncode:
      - RecResizeImg: {image_shape: [3, 48, 320]}
      - KeepKeys:
          keep_keys: [image, label_ctc, label_sar, length, valid_ratio]
  loader:
    shuffle: true
    batch_size_per_card: 128
    drop_last: true
    num_workers: 2

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: /content/data/data_vi/
    label_file_list: [/content/data/data_vi/nhan_val.txt]
    transforms:
      - DecodeImage: {img_mode: BGR, channel_first: false}
      - MultiLabelEncode:
      - RecResizeImg: {image_shape: [3, 48, 320]}
      - KeepKeys:
          keep_keys: [image, label_ctc, label_sar, length, valid_ratio]
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 128
    num_workers: 2
'''


## 5. Cấu hình

`character_dict_path` trỏ vào bảng 236 ký tự. Số lớp đầu ra đổi từ 185
lên 236 nên tầng cuối được khởi tạo lại, phần trích đặc trưng kế thừa.


In [ ]:
cfg = open('/content/rec_vi.yml', 'w', encoding='utf-8')
cfg.write(CFG_TEXT)
cfg.close()
!cp /content/rec_vi.yml configs/rec/rec_vi.yml
!tail -20 configs/rec/rec_vi.yml


## 6. Huấn luyện

T4 batch 128 chạy khoảng **25–40 phút một epoch** trên 200k ảnh. Colab
free hay ngắt sau ~4 tiếng, `save_epoch_step: 5` để còn chạy tiếp được.

Chạy tiếp sau khi bị ngắt: thêm
`-o Global.checkpoints=./output/rec_vi/latest`


In [ ]:
!python tools/train.py -c configs/rec/rec_vi.yml


## 7. Xuất ONNX


In [ ]:
!python tools/export_model.py -c configs/rec/rec_vi.yml \
  -o Global.pretrained_model=./output/rec_vi/best_accuracy \
     Global.save_inference_dir=./inference/rec_vi

!paddle2onnx --model_dir ./inference/rec_vi \
  --model_filename inference.pdmodel \
  --params_filename inference.pdiparams \
  --save_file /content/rec_vi.onnx \
  --opset_version 14 --enable_onnx_checker True
!ls -lh /content/rec_vi.onnx


## 8. Nghiệm thu

Đọc đúng dấu thì mới coi là xong. So với model Latin cũ trên cùng ảnh.


In [ ]:
import cv2
from rapidocr_onnxruntime import RapidOCR

moi = RapidOCR(rec_model_path='/content/rec_vi.onnx',
               rec_keys_path='/content/data/chars_vi.txt')
cu = RapidOCR()

import random
L = [x.split('\t') for x in open('/content/data/data_vi/nhan_val.txt', encoding='utf-8').read().splitlines() if x]
for f, that in random.sample(L, 10):
    im = cv2.imread('/content/data/data_vi/' + f)
    a, _ = moi(im)
    b, _ = cu(im)
    print('thật :', that)
    print('  mới:', [str(t[1]) for t in (a or [])])
    print('  cũ :', [str(t[1]) for t in (b or [])])
    print()


## 9. Mang về máy

Chép `rec_vi.onnx` và `chars_vi.txt` về, rồi trong `wrenautodub/ocr.py`
đổi chỗ khởi tạo:

```python
ocr = RapidOCR(rec_model_path='rec_vi.onnx',
               rec_keys_path='chars_vi.txt')
```

Lúc đó `_chon_dong` lấy được cả dòng tiếng Việt do NGƯỜI dịch, và bước
dịch bằng Google Translate có thể bỏ hẳn với phim hardsub song ngữ.


In [ ]:
!cp /content/data/chars_vi.txt /content/
!cd /content && tar -czf rapidocr_vi.tar.gz rec_vi.onnx chars_vi.txt
from google.colab import files
files.download('/content/rapidocr_vi.tar.gz')
